<a href="https://colab.research.google.com/github/A-J-Jovia/jovia-codeboosters-2026/blob/main/day7/day7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install chromadb sentence-transformers -q
import warnings
warnings.filterwarnings('ignore')

print("ChromaDB installed Successfully!!")

ChromaDB installed Successfully!!


In [4]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import chromadb

print("All libraries imported successfully!")
print(f"ChromaDB version: {chromadb.__version__}")

All libraries imported successfully!
ChromaDB version: 1.5.9


In [31]:
documents = [
    "ETL is data transformation pipeline",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automoblies",
    "Extracts, Transform and Load",
    "Machine learning trains models o data"
]

query_keyword = "vehicle"

print("="*60)
print(f"KEYWORD SEARCH for: {query_keyword}")
print("="*60)

for i,doc in enumerate(documents):
    if query_keyword.lower() in doc.lower():
      print((f" FOUND [doc_{i}]: {doc}"))
    else:
      print(f" MISSED [doc_{i}]: {doc}")

print()
print("PROBLEM:doc_2 talks about 'Cars and trucks' - which ARE vehicles!")
print("but keyword seach MISSED it because it searched for the exact word 'vehicle'.")

KEYWORD SEARCH for: vehicle
 MISSED [doc_0]: ETL is data transformation pipeline
 FOUND [doc_1]: A vehicle is a mode of transportation
 MISSED [doc_2]: Cars and trucks are popular automoblies
 MISSED [doc_3]: Extracts, Transform and Load
 MISSED [doc_4]: Machine learning trains models o data

PROBLEM:doc_2 talks about 'Cars and trucks' - which ARE vehicles!
but keyword seach MISSED it because it searched for the exact word 'vehicle'.


In [8]:
print("Loading Embedding model... (may take 1-2 minutes on first run)")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully")
print(f"Model produces vectors of size {model.get_sentence_embedding_dimension()} dimentions")

Loading Embedding model... (may take 1-2 minutes on first run)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully
Model produces vectors of size 384 dimentions


In [14]:
sentence = "ETL is used to clean and transform data"

embedding = model.encode(sentence)

print(f"Input sentence: {sentence}")
print(f"\nEmbedding Type: {type(embedding)}")
print(f"\nEmbedding Shape: {embedding.shape}")
print(f"\nFirst 10 Embeddings: {embedding[:10].round(4)}")
print(f"\nMin value: {embedding.min():.4f}")
print(f"\nMax value: {embedding.max():.4f}")

Input sentence: ETL is used to clean and transform data

Embedding Type: <class 'numpy.ndarray'>

Embedding Shape: (384,)

First 10 Embeddings: [-0.0784  0.0541  0.0224 -0.0389  0.0221 -0.0904  0.0007 -0.0152  0.0733
  0.0362]

Min value: -0.1381

Max value: 0.1815


In [30]:
# Using Dot product in numpy

print(f"Query: {query_keyword}\n")
for i, doc in enumerate(documents):
    print(f"Doc{i}: {doc}")
    embedding_query = model.encode(query_keyword.lower())
    embedding_doc = model.encode(doc.lower())
    similarity = np.dot(embedding_query, embedding_doc)

    if similarity > 0.4:
        print(f" FOUND [doc_{i}]: {doc} (similarity={similarity:.2f})\n")
    else:
        print(f" MISSED [doc_{i}]: {doc} (similarity={similarity:.2f})\n")


Query: vehicle

Doc0: ETL is data transformation pipeline
 MISSED [doc_0]: ETL is data transformation pipeline (similarity=0.04)

Doc1: A vehicle is a mode of transportation
 FOUND [doc_1]: A vehicle is a mode of transportation (similarity=0.73)

Doc2: Cars and rucks are popular automoblies
 FOUND [doc_2]: Cars and rucks are popular automoblies (similarity=0.47)

Doc3: Extracts, Transform and Load
 MISSED [doc_3]: Extracts, Transform and Load (similarity=0.09)

Doc4: Machine learning trains models o data
 MISSED [doc_4]: Machine learning trains models o data (similarity=0.19)



In [32]:
# Using cosine similarity in util module
from sentence_transformers import util

print(f"Query: {query_keyword}\n")
for i,doc in enumerate(documents):
  print(f"Doc{i}: {doc}")
  embedding_query = model.encode(query_keyword.lower())
  embedding_doc = model.encode(doc.lower())
  similarity = util.cos_sim(embedding_query, embedding_doc).item()
  if similarity > 0.4:
    print(f" FOUND [doc_{i}]: {doc} (similarity={similarity:.2f})\n")
  else:
    print(f" MISSED [doc_{i}]: {doc} (similarity={similarity:.2f})\n")

Query: vehicle

Doc0: ETL is data transformation pipeline
 MISSED [doc_0]: ETL is data transformation pipeline (similarity=0.04)

Doc1: A vehicle is a mode of transportation
 FOUND [doc_1]: A vehicle is a mode of transportation (similarity=0.73)

Doc2: Cars and trucks are popular automoblies
 FOUND [doc_2]: Cars and trucks are popular automoblies (similarity=0.52)

Doc3: Extracts, Transform and Load
 MISSED [doc_3]: Extracts, Transform and Load (similarity=0.09)

Doc4: Machine learning trains models o data
 MISSED [doc_4]: Machine learning trains models o data (similarity=0.19)



In [42]:
meanings =[
    "Thor is an Avenger.",
    "Avengers are saviours of the world. They save us from Bad people.",
    "Betty botter bought some butter.But she said the Butter's bitter"
]
sentence1 = "Thor holds Hammer in his hand. It can cause Thunder"
print(f"Query: {sentence1}\n")
for i,doc in enumerate(meanings):
  print(f"Doc{i}: {doc}")
  embedding_query = model.encode(sentence1.lower())
  embedding_doc = model.encode(doc.lower())
  similarity = util.cos_sim(embedding_query, embedding_doc).item()
  if similarity > 0.2:
    print(f" FOUND [doc_{i}]: {doc} (similarity={similarity:.2f})\n")
  else:
    print(f" MISSED [doc_{i}]: {doc} (similarity={similarity:.2f})\n")

Query: Thor holds Hammer in his hand. It can cause Thunder

Doc0: Thor is an Avenger.
 FOUND [doc_0]: Thor is an Avenger. (similarity=0.55)

Doc1: Avengers are saviours of the world. They save us from Bad people.
 FOUND [doc_1]: Avengers are saviours of the world. They save us from Bad people. (similarity=0.30)

Doc2: Betty botter bought some butter.But she said the Butter's bitter
 MISSED [doc_2]: Betty botter bought some butter.But she said the Butter's bitter (similarity=0.03)



In [36]:
sentences =[
    "ETL is used to clean and Transform data",
    "Data transformation is a key pipeline step",
    "The sky is blue and clouds are white."
]
embeddings = model.encode(sentences)

print(f"Number of sentence: {len(sentences)}")
print(f"Shape of Embeddings array: {embeddings.shape}\n")
print("Each row is one sentence's embegging")

for i,sent in enumerate(sentences):
  print(f"Sentence {i}: Shape:{embeddings[i].shape}, First 5 values={embeddings[i][:5].round(4)}")

Number of sentence: 3
Shape of Embeddings array: (3, 384)

Each row is one sentence's embegging
Sentence 0: Shape:(384,), First 5 values=[-0.0784  0.0541  0.0224 -0.0389  0.0221]
Sentence 1: Shape:(384,), First 5 values=[-0.0466  0.0585 -0.0012 -0.0334 -0.0461]
Sentence 2: Shape:(384,), First 5 values=[0.0584 0.0673 0.0652 0.061  0.043 ]


# **Using ChromaDB to store vectors**

In [43]:
chroma_client =chromadb.Client()
collection=chroma_client.get_or_create_collection("demo_notes")
print("ChromaDB client created (in-memory mode)")


print(f"Collection name:demo_notes ")
print(f"Documents in collection :{collection.count()}")

ChromaDB client created (in-memory mode)
Collection name:demo_notes 
Documents in collection :0


In [58]:
sample_docs = [
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automobiles",
    "SQL is used to query databases",
    "Machine learning trains models on data"
]

sample_ids = ['doc001', 'doc002', 'doc003', 'doc004', 'doc005']

sample_metadata = [
    {"subject": "Data Engineering", "topic": "ETL"},
    {"subject": "Data Engineering", "topic": "SQL"},
    {"subject": "Machine Learning", "topic": "ML Basic"},
    {"subject": "Python", "topic": "Pandas"},
    {"subject": "Machine Learning", "topic": "Neural Networks"}
]

sample_embeddings = model.encode(sample_docs).tolist()

collection.add(
    documents=sample_docs,
    embeddings=sample_embeddings,
    ids=sample_ids,
    metadatas=sample_metadata
)

print("Documents added successfully")
print(f"Number of documents in collection: {collection.count()}")

Documents added successfully
Number of documents in collection: 5


In [54]:
query="How do I clean and prepare data?"

results=collection.query(
    query_texts=[query],
    n_results=3
)

print("RESULT KEYS AVAILABLE:")
print(list(results.keys()))

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 31.8MiB/s]


RESULT KEYS AVAILABLE:
['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances']


In [59]:
print(f"Query")
print("="*60,'\n')

matched_docs = results['documents'][0]
matched_ids = results['ids'][0]
matched_distances = results['distances'][0]
matched_metadata = results['metadatas'][0]

for rank, (doc, doc_id, dist, meta) in enumerate(zip(matched_docs, matched_ids, matched_distances, matched_metadata)):
  print(f"Rank: {rank} | ID: {doc_id} | Distance: {dist:.2f}")
  print(f"Subject: {meta['subject']} | Topic: {meta['topic']}")
  print(f"Document:{doc}\n")


Query

Rank: 0 | ID: doc001 | Distance: 1.62
Subject: Data Engineering | Topic: ETL
Document:ETL is data transformation pipeline

Rank: 1 | ID: doc005 | Distance: 1.70
Subject: Machine Learning | Topic: Models
Document:Machine learning trains models o data

Rank: 2 | ID: doc004 | Distance: 1.71
Subject: Data Engineering | Topic: Data Transformation
Document:Extracts, Transform and Load



In [61]:
filtered_results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"subject": "Data Engineering"}
)



print(f"FILTERED QUERY: {query}")
print("Filter only Machine Learning documents")
print("="*60)

for rank, (doc, dist, meta) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['distances'][0],
    filtered_results['metadatas'][0]),start=1):
  print(f"Rank: {rank} | Distance: {dist:.2f} | Subject: {meta['subject']}")
  print(f"{doc}\n")

FILTERED QUERY: How do I clean and prepare data?
Filter only Machine Learning documents
Rank: 1 | Distance: 1.62 | Subject: Data Engineering
ETL is data transformation pipeline

Rank: 2 | Distance: 1.71 | Subject: Data Engineering
Extracts, Transform and Load



In [62]:
print("DISTANCE TO SIMILARITY CONVERSION")
print(f"{'Distance':<15}{'Similarity':<15}{'Interpretation':<20}")
distances=[0.05,0.20,0.40,0.65,0.90]
interpretations=["Near identical","Very similar","Related","Somewhat related","Not related"]
for dist,interp in zip(distances,interpretations):
  similarity=1-dist
  print(f"{dist:<15.2f}{similarity:<15.2f}{interp:<20}")

DISTANCE TO SIMILARITY CONVERSION
Distance       Similarity     Interpretation      
0.05           0.95           Near identical      
0.20           0.80           Very similar        
0.40           0.60           Related             
0.65           0.35           Somewhat related    
0.90           0.10           Not related         
